# Test de Sistemas Cuánticos

Se tiene como primeros objetivos:
- Tomar un potencial y construir la ecuación de Schödinger.
- Resolver la ecuación con distintas estrategias dependiendo del tipo de potencial.
- Aplicar las condiciones de frontera a la solución general.
- Cuantizar la energía y calcular estos valores permitidos, así como la energía del estado fundamental
- Obtener un estado de energía $n$ y regresar una instancia de la clase `WaveFunction` caraterística $\psi_{n}(x)$

Para comenzar, resolveremos sistemas cuánticos sencillos que impliquen potenciales sencillos como el del pozo de potencial infinito, pero sin resolver dicho potencial a trozos, si no tomar solo la parte que nos interesa.

In [ ]:
import sympy as sp
import matplotlib.pyplot as plt
import numpy as np

sp.init_printing()

x = sp.symbols("x", real=True)
E = sp.symbols("E", positive=True)
C1, C2, L, m, hbar = sp.symbols("C_1 C_2 L m hbar", positive=True)

psi = sp.Function("psi")

## Pozo de potencial infinito

Supondremos un potencial $V(x) = 0$ en un dominio de $[0, L]$

In [ ]:
V = sp.S.Zero
T = - hbar**2 / (2*m) * sp.diff(psi(x), x, 2)

H = T + V
ecuación = sp.Eq(E * psi(x), H)
ecuación

Este es un potencial que puede ser resuelto facilmente con `sp.dsolve()`, y miremos primero como queda la ecuación general y luego al aplicar las condiciones de frontera como argumento con `ics=`, las cuales son:

$$
\psi (0) = 0, \psi (L) = 0
$$

In [ ]:
solución_general = sp.dsolve(ecuación, psi(x))
display(solución_general)

solución_general_ics = sp.dsolve(ecuación, psi(x), ics={psi(0):0, psi(x).subs(x, L):0})
display(solución_general_ics)

Como podemos ver, al aplicar las condiciones de frontera directamente en `sp.dsolve()` el método toma directamente la solución trivial, por lo tanto, tendremos que aplicar estas condiciones de frontera de manera manual.

Para ello, tomaremos el lado derecho de la solución general y extraeremos los símbolos, particularmente las constantes, de esta y les asignaremos sus correspondientes suposiciones. Pero primero simplifiquemos algunas constantes mediante $k = \frac{\sqrt{2mE}}{\hbar}$ :

In [ ]:
expr = solución_general.rhs

# Simplificamos
k = sp.Symbol('k', positive=True)
k_expr = sp.sqrt(2*m*E)/hbar
expr = expr.subs(k_expr, k)

constantes_originales = expr.free_symbols - {x, E, hbar, m}

In [ ]:
# Dado que devuelve un conjunto, los transformamos a lista
constantes_originales = list(sorted(constantes_originales, key=str)) # Ordenamos para asegurarnos que esten ordenadas

# Añadimos las suposiciones a las constantes con el mismo nombre de las extraidas
C1 = sp.Symbol(constantes_originales[0].name, real=True, nonzero=True)
C2 = sp.Symbol(constantes_originales[1].name, real=True)
constantes = sorted([C1, C2], key=str)

expr = expr.subs({
    constantes_originales[0]: constantes[0],
    constantes_originales[1]: constantes[1]
})

Y volvemos a tener la misma solución general pero con suposiciones asignadas a las constantes, de modo que no debería devolvernos la solución trivial. 

Tratemos de obtener nuevamente la solución aplicando las condiciones de forntera.

In [ ]:
# Ecuaciones a resolver
condiciones = [
    sp.Eq(expr.subs(x, 0), 0, evaluate=False),
    sp.Eq(expr.subs(x, L), 0, evaluate=False)
]

condiciones

In [ ]:
constantes_sol = sp.solve(
    condiciones,
    constantes,
    dict=True
)

constantes_sol

Como podemos ver, nos devuelve una lista vacía. Esto se debe a que SymPy aún trata de devolvernos la solución trivial pero ahora se contradice con las suposiciones que hicimos a las constantes, particularmente $C_1 \neq 0$.

Entonces, trataremos de aplicar dichas condiciones de frontera mediante un sistema de ecuaciones y calculando los eigenvalores y eigenvectores.

En primer lugar, para crear la matriz de funciones sin las constantes, aplicaremos la derivada a la ecuación sobre estas de modo que obtengamos solo los coeficientes de estas, es decir, para que estas se eliminen y nos quede una matriz sin las constantes.

In [ ]:
A = sp.Matrix([
    [sp.diff(condicion.lhs, constante) for constante in constantes] for condicion in condiciones
])
# Observemos que se tomó el lado izquierdo pues condiciones es una lista de ecuaciones

A

Creamos la matriz de constantes y ya tendríamos nuestro sistema de ecuaciones.

In [ ]:
C = sp.Matrix([
    C1,
    C2
])

display(sp.Eq(A * C, 0, evaluate=False))

Ahora sí, si el determinante es distinto de cero entonces podemos encontrar soluciones no triviales. Asi que primero encontremos los valores de k permitidos y de ahí sustituimos E para encontrar los valores de energía permitidos, recordando que:
$$
k^2 = \frac{2 m E}{\hbar ^2}, E = \frac{\hbar ^2 k ^2}{2m}
$$

In [ ]:
n = sp.symbols('n', integer=True, positive=True)
det_A = sp.simplify(A.det())

if sp.simplify(det_A) != 0:
    sol_k = sp.solveset(
    det_A,
    k,
    domain=sp.S.Reals
)

sp.simplify(sol_k)

Conociendo ahora que `sp.solveset()` regresa un objeto de SymPy que se conoce como ImageSet y entgendiendo que tiene diferentes propiedades para poder acceder a los diferentes valores del conjunto, aplicamos el siguiente código:

In [ ]:
N = 4
#sol_k.args[0].lamda.variables[0]
k_valores = set()
for rama in sol_k.args:
    display(rama)
    j = rama.lamda.variables[0]
    for i in range(-N, N+1):
        valor = sp.simplify(rama.lamda.expr.subs(j, i))
        print(f"i={i}, valor={valor}")
        if valor.is_positive:
            print(f"Agregando valor: {valor}")
            k_valores.add(valor)


orden = lambda v: float(v.subs({s: 1 for s in v.free_symbols}))
k_valores = sorted(k_valores, key=orden)[:N]

Y así es como obtenemos un conjunto con los valores de la variable k que son solución al problema $sin(kL) = 0$ y, por lo tanto, ahora podemos sustituirlos en la energía para cuantizarla y obtener finalmente los valores permitidos de la energía, los cuales de hecho, son exactamente los mismos a los que se obtienen de la ecuación ya conocida:

$$
E_n = \frac{n^2 \pi ^2 \hbar ^2}{2 m L^2}
$$

In [ ]:
k_n = sp.Symbol("k_n", positve=True)

# Despejamos la expresión de E en función de la expresión de k que ya conocíamos
E_expr = sp.solve(k_n - k_expr, E)[0]
display(sp.Eq(sp.Symbol("E_n"), E_expr))

E_n = [E_expr.subs(k_n, k_i) for k_i in k_valores]
E_n[:N]

Con los niveles de energía, ahora podemos obtener las funciones de onda permitidas. Para ello usaremos la solución general con la que se incició la resolución del pozo de potencial con la primer condición de frontera aplicada $C_2 = 0$ y luego se sustituirán los valores $k_n$ obtenidos.

In [ ]:
# Recordemos algunas cosas
# Aplicamos la primer condición C2 = 0 a la solución general
display(condiciones[0])
display(expr)

# Obtenemos la expresión de cada estado del sistema
psi_expr = expr.subs({C2 : condiciones[0].rhs, k : k_n})
display(sp.Eq(sp.Symbol("psi_n"), psi_expr))

# Sustituimos cada valor k para obtener diferentes funciones
psi_n = []
for k_i in k_valores:
    psi_n.append(psi_expr.subs(k_n, k_i))

psi_n[:N]

Con la expresión de las funciones de onda $\psi _n$, solo queda graficar un par con su respectivo nivel de energía para observar graficamente el resultado :)

In [ ]:
# Constantes a sustituir (luego de normalizar)
cte = {L:1, C1:sp.sqrt(2)}

# Con desplazamientos verticales provisionales que hacen alusión a su nivel de energía
psi_1 = sp.lambdify(x, psi_n[0].subs(cte) +1, "numpy")
psi_2 = sp.lambdify(x, psi_n[1].subs(cte) +5, "numpy")

posiciones = np.linspace(0, 1, 100)

fig, ax = plt.subplots()
ax.plot(posiciones, psi_1(posiciones), color="blue", label="psi_1(x)")
ax.plot(posiciones, psi_2(posiciones), color="red", label="psi_2(x)")
ax.hlines(1, xmin=0, xmax=1, color="blue", linestyle="--", linewidth=1)
ax.hlines(5, xmin=0, xmax=1, color="red", linestyle="--", linewidth=1)
ax.set_xlabel("x")
ax.set_ylabel("E")
ax.set_title("Pozo de potencial infinito")
ax.legend()
ax.grid()
fig.savefig("./images/prueba_pozo_potencial.jpg")
plt.show()

## Futuras implementaciones

A continuación algunos puntos los cuales serían buenos agregar, para ello se deberá investigar y justificar su uso.

- ``picewise()`` para funciones a trozos
- argumento ``hint=`` en ``dsolve()`` para ayudar a encontrar la solucion
- Coeficientes de transmision y reflexion para sistemas de dispersión
- Densidad de corriente de probabilidad $J(x) = \frac{\hbar}{m} Im(\psi^* \frac{d\psi}{dx})$
donde se usará ``sp.im()`` para extraer la parte imaginaria